# Horizon runtime on service_tickets (scalability)

Run the real Horizon repair pipeline on increasing row counts of
`service_tickets_21m/clean.csv` and time it — the §6.4 / Figure 5c–d runtime
experiment.

`service_tickets` has no injected errors or ground truth, so there is **no
precision/recall** here — only runtime. We cut a subset CSV per size from the
head of `clean.csv` (streamed, so the 14.6 GB file is read once up to the
largest size), then for each subset run `FDPatternGraph` → `repair_dirty_data`
and record graph-build, repair, and total seconds.

**Inputs:** `datasets/service_tickets_21m/clean.csv` + its FDs → **Outputs:** a
per-size timing table and a log-log runtime plot in `output/`.

Config knobs (see the config cell):
- `DATABASE` — the dataset under `datasets/` (`service_tickets_21m`).
- `SIZES` — the row counts to time; committed as the two large points
  `[1_000_000, 21_000_000]` (1M and the full 21M dataset).

In [1]:
import importlib.util
import logging
import sys
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless: get_ordered_fds renders FD/SCC PNGs, no GUI backend
import matplotlib.pyplot as plt
import polars as pl

# horizon/ pipeline modules use flat imports and need horizon/ on sys.path; eval
# uses package imports (horizon.*) and needs the repo root. Put both on the path,
# repo root first, and load horizon/horizon.py by file path under a private name to
# dodge the package/module name clash on `horizon`. See repair_eval.ipynb.
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

spec = importlib.util.spec_from_file_location("horizon_pipeline", HZN / "horizon.py")
pipe = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipe)

from fd_pattern_graph import FDPatternGraph
from static_fd_analysis import get_ordered_fds
from horizon.utils.loaders import load_fds, load_table

DATASETS = REPO / "datasets"
OUTPUT = REPO / "output"
OUTPUT.mkdir(exist_ok=True)

logging.getLogger("horizon").setLevel(logging.WARNING)  # mute per-tuple progress logs

In [ ]:
# >>> the two knobs <<<
DATABASE = "service_tickets_21m"
SIZES = [1_000_000, 21_000_000]  # full dataset is 21M rows

In [ ]:
# Cut a subset CSV per size from the head of clean.csv, REUSING any that already
# exist so re-runs don't rewrite them. clean.csv is read only if some subset is
# missing, and only up to the largest missing size (polars stops parsing after
# n_rows, so the 14.6 GB file is never fully read unless a full-size subset is
# missing). infer_schema_length=0 -> all columns as strings, matching how the
# pipeline loads data (load_table -> Utf8).
ds_dir = DATASETS / DATABASE
clean_csv = ds_dir / "clean.csv"
subset_dir = OUTPUT / f"{DATABASE}_subsets"
subset_dir.mkdir(parents=True, exist_ok=True)

subset_paths = {n: subset_dir / f"clean_{n}.csv" for n in SIZES}

missing = [n for n in SIZES if not subset_paths[n].exists()]
if missing:
    base = pl.read_csv(clean_csv, n_rows=max(missing), infer_schema_length=0)
    for n in missing:
        base.head(n).write_csv(subset_paths[n])

for n in SIZES:
    print(f"{subset_paths[n].name}: {n:,} rows ({'written' if n in missing else 'reused'})")

In [ ]:
# FDs + traversal order depend only on the schema -> compute once. Per size,
# rebuild the pattern graph and run the repair, timing each stage separately.
# repair_secs includes the in-repair data load (repair_dirty_data re-reads the
# subset), same as in repair_eval. collect_pattern_expressions=False: the lineage
# is discarded here (`_`), so don't retain one PatternExpression per tuple.
set_of_fds = load_fds(ds_dir, clean_csv)
ordered_fds = get_ordered_fds(set_of_fds, DATABASE, OUTPUT)[0]

rows = []
for n in SIZES:
    path = subset_paths[n]

    t0 = time.perf_counter()
    graph = FDPatternGraph(path, set_of_fds)
    t_graph = time.perf_counter() - t0

    t1 = time.perf_counter()
    cleaned_out = OUTPUT / f"{DATABASE}_cleaned_{n}.csv"
    cleaned = pipe.repair_dirty_data(
        path, cleaned_out, ordered_fds, graph.repair_table, graph, collect_pattern_expressions=False
    )[0]
    t_repair = time.perf_counter() - t1

    rows.append(
        {
            "n_rows": n,
            "graph_secs": round(t_graph, 2),
            "repair_secs": round(t_repair, 2),
            "total_secs": round(t_graph + t_repair, 2),
        }
    )
    print(f"{n:>9,} rows  graph={t_graph:7.2f}s  repair={t_repair:8.2f}s  total={t_graph + t_repair:8.2f}s")
stats = pl.DataFrame(rows)
display(stats)

In [ ]:
# log-log runtime vs #tuples (Figure 5c-d style). Saved to OUTPUT and shown
# inline as a PNG, since the Agg backend won't render figures directly.
from IPython.display import Image

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(stats["n_rows"], stats["total_secs"], "o-", label="total")
ax.plot(stats["n_rows"], stats["graph_secs"], "s--", label="graph build")
ax.plot(stats["n_rows"], stats["repair_secs"], "^--", label="repair")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("#tuples")
ax.set_ylabel("runtime (sec)")
ax.set_title(f"Horizon runtime — {DATABASE}")
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.legend()
fig.tight_layout()

plot_path = OUTPUT / f"{DATABASE}_runtime.png"
fig.savefig(plot_path, dpi=120)
plt.close(fig)
Image(str(plot_path))